# Implementing an Earley Parser (Chatty Version)

This notebook implements the same Earley parser as the notebook `01-Earley-Parser.ipynb`, but in addition it shows
how the parser proceeds:  Before the parser processes the set $Q_i$, it prints the next token.  Every *Earley object*
that is added by one of the operations *completion*, *prediction*, or *scanning* is printed together with the name of
the operation.  Finally, the sets $Q_i$ and $Q_{i+1}$ are printed.

Furthermore, the parser remembers for every *Earley object* all the ways in which it has been created.  If parsing
is successful, this information is used to compute all *parse trees* of the input, which are drawn with
`Graphviz`.  If the grammar is ambiguous, a string can have several parse trees.  While the notebook
`01-Earley-Parser.ipynb` only *recognizes* whether a string is generated by the grammar, this notebook really
*parses* the string.  Only the class `EarleyParser` and its methods, the cells computing the parse trees, and the
function `test` differ from the notebook `01-Earley-Parser.ipynb`.

## Setup and Imports

We import the module `re`, which is used by the function `tokenize`, and `Larkita`.  The parsers generated by
`Larkita` return the abstract syntax tree of their input as a nested tuple.  The notebook `AST2Dot.ipynb` provides
the function `tuple2dot`, which uses `Graphviz` to draw a tree that is represented as a nested tuple.

In [ ]:
import re

from IPython.display import Code, display

from larkita import Larkita

In [ ]:
%run ../AST2Dot.ipynb

The type checker cannot follow the command `%run`.  Therefore, the next cell declares the function `tuple2dot`
and the type `AST` of the notebook `AST2Dot.ipynb` for the type checker.  This cell is never executed.

In [ ]:
from typing import TYPE_CHECKING
if TYPE_CHECKING:  # declarations for the type checker, never executed
    import graphviz as gv
    type AST = str | int | float | tuple[str, *tuple["AST", ...]]
    def tuple2dot(t: AST) -> gv.Digraph: ...

The file `simple.g` contains a grammar describing arithmetic expressions that only use the operators `+` and `*`.

In [ ]:
Code(filename='simple.g', language='larkita')

## Specification of the Grammar

`Larkita` allows us to define both lexical tokens and structural rules natively in a single file written in
*extended Backus-Naur Form* (EBNF).  Below is the grammar that `Larkita` will use to parse our context-free grammar specifications
(like `simple.g`).  These specifications are written in the notation of `Larkita`, too:  The alternatives of a
rule are separated by `"|"`, and a rule is not terminated by a semicolon.
* The token `VARIABLE` recognizes a *syntactical variable*.  Syntactical variables start with a lower case letter.
* The token `RULE_NAME` recognizes the variable that is defined by a rule, i.e. a variable that is followed by a
  colon.  After a rule, a variable can either be the next item of this rule or the beginning of the next rule, and
  only the colon tells these cases apart.  As an LR(1) parser sees only one token ahead, the scanner has to make this
  distinction:  The *lookahead* `(?=[ \t]*:)` at the end of the regular expression checks that a colon follows
  without making it part of the token.
* The token `TERMINAL` recognizes a *token*.  Tokens start with an uppercase letter.
* The token `LITERAL` recognizes a non-empty sequence of characters that is enclosed in double quotes.
* The string literals of an alternative form the head of the tuple that is returned for this alternative.  As the
  literals `":"` and `"|"` carry no information, the annotation `%ignore ":", "|"` removes them.  Then no literal is
  left and the name of the rule is used as head:  A rule yields the tuple `('rule', variable, body_1, ..., body_n)`,
  and a body yields the tuple `('body', item_1, ..., item_n)`.
* An alternative that consists of a single symbol returns the abstract syntax tree of this symbol.  Therefore, the
  rule `item` creates no tuple of its own.  A token is represented as a pair consisting of the name of the token and
  its text, for example `('LITERAL', '"+"')`.
* The `%ignore` directive tells the scanner to discard whitespace and newlines so they do not clutter our syntax tree.

In [ ]:
Code(filename='meta-grammar.g', language='larkita')

We generate the parser.  By default, `Larkita` generates a *canonical LR(1) parser* and writes a description of its
states into the file `meta-grammar.out`.

In [ ]:
grammar_parser = Larkita.open('meta-grammar.g', out_file='meta-grammar.out')

Let us check that Larkita was indeed able to generate a parser.

In [ ]:
with open('meta-grammar.out') as handle:
    report = handle.read()
print(report[:report.index('States\n')])

## Generating the Abstract Syntax Tree (AST)

The parser returns a nested tuple of the form `('start', rule_1, ..., rule_n)`.  Our custom `EarleyParser` expects the
grammar to be a list of rules, where each rule is a list of strings (e.g., `['expr', 'expr', '"+"', 'product']`).
Every token in the abstract syntax tree is a pair consisting of the name of the token and its text, and only the
text is needed.  The literals keep their double quotes, so that a literal like `"e"` can never be confused with
a variable like `e`.

The function `ast_to_rules` drops the heads `'start'`, `'rule'`, and `'body'`:  For every body of a rule defining
the variable `a`, it creates the list that starts with `a` and continues with the items of the body.  Of every
token, only its text is kept.

In [ ]:
def ast_to_rules(ast: tuple) -> list[list[str]]:
    _, *rules = ast
    return [ [variable] + [text for _, text in items]
             for _, (_, variable), *bodies in rules
             for _, *items in bodies ]

The function `read_grammar` reads the grammar file, displays it, parses it into a nested tuple, and converts it
into a list of rules.

In [ ]:
def read_grammar(file):
    with open(file, 'r') as handle:
        grammar_str = handle.read()
    display(Code(grammar_str, language='larkita'))
    ast = grammar_parser.parse(grammar_str)
    return ast_to_rules(ast)

In [ ]:
read_grammar('simple.g')

## Earley's Algorithm

Given a context-free grammar $G = \langle V, \Sigma, R, s \rangle$ and a string $w = x_1x_2 \cdots x_n \in \Sigma^*$ of length $n$, 
an *Earley object* is a pair of the form
$$\langle a \rightarrow \alpha \bullet \beta, k \rangle$$
such that 
- $(a \rightarrow \alpha \beta) \in R\quad$  and
- $k \in \{0,1,\cdots,n\}$. 

In the literature, Earley objects are also called *Earley items*.

The class `EarleyObject` represents a single *Earley object*.  It is a *data class*, i.e. a class whose main
purpose is to store data.  A class becomes a data class if its definition is preceded by the *decorator*
`@dataclass`, which is imported from the module `dataclasses` of the standard library.  In the body of a data class,
we only declare the *fields* of the class, i.e. its member variables, together with their types:
- `mVariable` is the variable $a$,
- `mAlpha` is $\alpha$, a tuple of strings,
- `mBeta` is $\beta$, a tuple of strings, and
- `mIndex` is $k$.

From these declarations, the decorator `@dataclass` generates the following methods:
- The *constructor* `__init__` has one parameter for every field.  The parameters occur in the order in which the
  fields are declared, and the constructor stores the arguments in the fields.  For example, the call
  `EarleyObject('expr', ('expr', '"+"'), ('product',), 0)` creates the Earley object
  $\langle \texttt{expr} \rightarrow \texttt{expr}\; \texttt{"+"} \bullet \texttt{product}, 0 \rangle$.
- The method `__eq__` compares two objects field by field:  Two Earley objects are equal if and only if they
  agree in all four fields.  Python derives the operator `!=` from the method `__eq__`.

The arguments of the decorator control which further methods are generated:
- `frozen=True` makes the objects *immutable*:  After an Earley object has been created, its fields cannot be
  changed.  An assignment to a field raises the exception `FrozenInstanceError`.  As the fields of a frozen
  object never change, its hash value cannot change either.  Therefore, the decorator also generates the method
  `__hash__`, which computes the hash value of an object from the values of all its fields.  We need this method,
  since Earley's algorithm stores Earley objects in sets and Python requires the elements of a set to have a hash
  value.  For the same reason, `mAlpha` and `mBeta` are tuples:  Lists can be changed and therefore have no hash
  value.
- By default, the decorator also generates a method `__repr__`, which lists the names and the values of all
  fields.  The argument `repr=False` suppresses this method, since we define our own method `__repr__`, which
  writes an Earley object in the notation used above.

Without the decorator, we would have to write the constructor and the methods `__eq__` and `__hash__` ourselves.

The methods that are only declared in the class body, i.e. with the body `...`, are implemented in the
cells below and attached to the class afterwards.  These declarations tell the type checker which
methods the class provides.

In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True, repr=False)
class EarleyObject():
    mVariable: str
    mAlpha:    tuple[str, ...]
    mBeta:     tuple[str, ...]
    mIndex:    int

    def __repr__(self) -> str:
        rhs = ' '.join(self.mAlpha + ('•',) + self.mBeta)
        return f'<{self.mVariable} → {rhs}, {self.mIndex}>'

    def isComplete(self) -> bool: ...
    def sameVar(self, c: str) -> bool: ...
    def scan(self, T: str) -> bool: ...
    def nextVar(self) -> str | None: ...
    def moveDot(self) -> EarleyObject: ...

Given an Earley object `self`, the method `isComplete` checks whether the Earley object `self` has the form
$$\langle a \rightarrow \alpha \bullet, k \rangle,$$
i.e. whether the $\bullet$ is at the end of the grammar rule.

In [ ]:
def isComplete(self):
    return self.mBeta == ()

EarleyObject.isComplete = isComplete
del isComplete

The method `sameVar(self, c)` checks whether the symbol following the dot is the same as the variable 
given as argument, i.e. `sameVar(self, c)` returns `True` if `self` is an Earley object of the form
$$\langle a \rightarrow \alpha \bullet c\beta, k \rangle.$$

In [ ]:
def sameVar(self, c: str) -> bool:
    return len(self.mBeta) > 0 and self.mBeta[0] == c

EarleyObject.sameVar = sameVar 
del sameVar

The method `scan(self, T)` checks whether the symbol following the dot matches the token `T`, 
i.e. `scan(self, T)` returns `True` if `self` is an Earley object of the form
$$\langle a \rightarrow \alpha \bullet T\beta, k \rangle.$$
The argument $T$ can either be the name of a token or a literal.

In [ ]:
def scan(self, T: str) -> bool:
    return len(self.mBeta) > 0 and self.mBeta[0] == T

EarleyObject.scan = scan
del scan

Given an Earley object, the method `nextVar` returns the name of the variable following the dot.  If there is no variable following the dot, the method returns `None`.  The method can distinguish variables from tokens because only the names of variables start with a lower case letter:  The names of tokens start with an upper case letter and literals start with a double quote.

In [ ]:
def nextVar(self) -> str | None:
    if len(self.mBeta) > 0:
        var = self.mBeta[0]
        if var[0].islower():
            return var
    return None

EarleyObject.nextVar = nextVar
del nextVar

The method `moveDot(self)` moves the $\bullet$ in the Earley object `self`, where `self` has the form 
$$\langle a \rightarrow \alpha \bullet \beta, k \rangle$$
over the next variable, token, or literal in $\beta$.  It assumes that $\beta$ is not empty.

In [ ]:
def moveDot(self) -> EarleyObject:
    return EarleyObject(self.mVariable, 
                        self.mAlpha + (self.mBeta[0],), 
                        self.mBeta[1:], 
                        self.mIndex)

EarleyObject.moveDot = moveDot
del moveDot

The class `Grammar` represents a context-free grammar.  It stores a list of the rules of the grammar.
Each grammar rule of the form
$$ a \rightarrow \beta $$
is stored as a list that starts with $a$ and continues with the symbols of $\beta$.  The start symbol is assumed to be the variable on the left hand side of
the first rule. To distinguish syntactical variables from tokens, variables start with a lower case letter,
while tokens either start with an upper case letter or they are literals like `"+"`, which keep their double
quotes.

The methods that are only declared in the class body, i.e. with the body `...`, are implemented in the
cells below and attached to the class afterwards.  These declarations tell the type checker which
methods the class provides.

In [ ]:
class Grammar():
    def __init__(self, Rules: list[list[str]]):
        self.mRules = Rules

    def startObject(self) -> EarleyObject: ...
    def finishObject(self) -> EarleyObject: ...
    def startVar(self) -> str: ...

The method `startObject` returns the Earley object
$$ \langle\hat{s} \rightarrow \bullet s, 0\rangle $$
where $s$ is the start variable of the given grammar and $\hat{s}$ is the new variable `Start`.

In [ ]:
def startObject(self):
    return EarleyObject('Start', (), (self.startVar(),), 0)

Grammar.startObject = startObject
del startObject

The method `finishObject` returns the Earley object
$$ \langle\hat{s} \rightarrow s \bullet, 0\rangle $$
where $s$ is the start variable of the given grammar and $\hat{s}$ is a new variable.

In [ ]:
def finishObject(self):
    return EarleyObject('Start', (self.startVar(),), (), 0)

Grammar.finishObject = finishObject
del finishObject

The method `startVar` returns the start variable of the grammar.  It is assumed that
the first rule of the grammar defines the start variable.

In [ ]:
def startVar(self) -> str:
    return self.mRules[0][0]

Grammar.startVar = startVar
del startVar

In order to compute the parse trees, the parser remembers for every *Earley object* all the ways in which it has
been created.  Every way is described by a *back pointer* of type `BackPointer`:
- If the *Earley object* $\langle a \rightarrow \beta T \bullet \gamma, k\rangle \in Q_{i+1}$ has been created by
  *scanning* the token $T$, the back pointer is the triple
  $$ \bigl(\langle a \rightarrow \beta \bullet T \gamma, k\rangle, i, T\bigr). $$
- If the *Earley object* $\langle a \rightarrow \beta c \bullet \delta, k\rangle \in Q_i$ has been created by
  *completion* from the *Earley objects* $\langle a \rightarrow \beta \bullet c \delta, k\rangle \in Q_j$ and
  $\langle c \rightarrow \gamma \bullet, j\rangle \in Q_i$, the back pointer is the triple
  $$ \bigl(\langle a \rightarrow \beta \bullet c \delta, k\rangle, j, \langle c \rightarrow \gamma \bullet, j\rangle\bigr). $$

In both cases, the first component is the *Earley object* whose marker has been moved, the second component is the
index of the set that contains this *Earley object*, and the third component describes the symbol that the marker has
been moved over:  It is either the token itself or the complete *Earley object* that has recognized the variable.
Every *Earley object* is mapped to the list of its back pointers.  This list is empty for the start object
$\langle\hat{s} \rightarrow \bullet s, 0\rangle$ and for the *Earley objects* that have been created by *prediction*,
since the marker is at the beginning of their right hand side.  If the grammar is ambiguous, the list can contain
several back pointers.

A *parse tree* is represented as a nested tuple of type `ParseTree`:  A token is a leaf of the tree.  A variable $a$
that has been parsed with the grammar rule $a \rightarrow X_1 \cdots X_n$ is represented as the tuple
`(a, t_1, ..., t_n)`, where `t_i` is the parse tree of $X_i$.  If $a$ has been parsed with a $\lambda$-rule, i.e. if
$n = 0$, it is represented as the pair `(a, 'λ')`.  Every parse tree is an abstract syntax tree in the sense of the
notebook `AST2Dot.ipynb`, so it can be drawn with the function `tuple2dot`.

The type `Active` is explained below, where the parse trees are computed.

In [ ]:
type BackPointer = tuple[EarleyObject, int, EarleyObject | str]
type ParseTree   = str | tuple[str, *tuple[ParseTree, ...]]
type Active      = frozenset[tuple[EarleyObject, int]]

The class `EarleyParser` implements the [parsing algorithm of Jay Earley](https://en.wikipedia.org/wiki/Earley_parser).
The class maintains the following member variables:
- `mGrammar` is the grammar that is used to parse the given token string.
- `mString` is the list of tokens and literals that has to be parsed.

   As a hack, the first element of this list is `None`.  
   Therefore, `mString[i]` is the $i^\textrm{th}$ token.
- `mStateList` is a list of sets of *Earley objects*.  If $n$ is the length of the given token string
  (excluding the first element `None`), then $Q_i = \texttt{mStateList}[i]$. 
  The idea is that the set $Q_i$ is the set of those *Earley objects* that the parser could be in 
  when it has read the tokens `mString[1]`, $\cdots$,  `mString[i]`.  $Q_0$ is initialized as follows:
  $$ Q_0 = \bigl\{\langle\hat{s} \rightarrow \bullet s, 0\rangle\bigr\}. $$

  The sets $Q_i$ are stored as dictionaries that map every *Earley object* to the list of its back pointers.  In
  addition, a dictionary remembers the order in which its keys have been inserted.  Therefore, the *Earley objects*
  are printed in the order in which they have been added, while the order in which the elements of a set are
  printed depends on their hash values, and these change every time the notebook is restarted.
  
The *Earley objects* are interpreted as follows: If we have
$$ \langle c \rightarrow \alpha \bullet \beta, k\rangle \in Q_i, $$
then we know the following:
- After having read the tokens `mString[:k+1]` the parser tries to parse the variable $c$
  in the token string `mString[k+1:]`.
- After having read the token string `mString[k+1:i+1]` the parser has already recognized $\alpha$
  and now needs to recognize $\beta$ in the token string `mString[i+1:]` in order to parse the variable $c$.

The methods that are only declared in the class body, i.e. with the body `...`, are implemented in the
cells below and attached to the class afterwards.  These declarations tell the type checker which
methods the class provides.

The constructor prints the list of tokens.

In [ ]:
class EarleyParser():
    def __init__(self, grammar, TokenList):
        self.mGrammar   = grammar
        self.mString    = [None] + TokenList  # hack so mString[1] is the first token
        self.mStateList: list[dict[EarleyObject, list[BackPointer]]]
        self.mStateList = [{} for _ in range(len(TokenList)+1)]
        self.mStateList[0] = { self.mGrammar.startObject(): [] }
        print(f'Input: {TokenList}')

    def parse(self)            -> bool: ...
    def complete(self, i: int) -> bool: ...
    def predict(self, i: int)  -> bool: ...
    def scan(self, i: int)     -> None: ...
    def parseTrees(self)       -> list[ParseTree]: ...
    def trees(self, obj: EarleyObject, i: int, active: Active)    -> list[ParseTree]: ...
    def children(self, obj: EarleyObject, i: int, active: Active) -> list[list[ParseTree]]: ...

The method `parse` implements Earley's algorithm.  For all sets 
$Q_0$, $Q_1$, $\cdots$, $Q_n$ we proceed as follows:
- We apply the *completion* operation followed by the *prediction* operation.
  This is done until no more Earley objects are added to $Q_i$.  
  
  (The inner `while` loop is not necessary if the grammar does not contain $\lambda$-rules.)
- Finally, the *scanning* operation is applied to $Q_i$.  This operation adds
  Earley objects to the set $Q_{i+1}$.

After $Q_i$ has been computed, we proceed to process $Q_{i+1}$.
Parsing is successful iff
$$ \langle\hat{s} \rightarrow s \bullet, 0\rangle \in Q_n. $$
The method returns `True` if parsing is successful and `False` otherwise.

Before the set $Q_i$ is processed, the method `parse` prints the next token.  After the scanning operation, it
prints the sets $Q_i$ and $Q_{i+1}$.

In [ ]:
def earley_parse(self) -> bool:
    "run Earley's algorithm"
    n = len(self.mString) - 1 # mString[0] = None
    for i in range(0, n+1):
        if i + 1 <= n:
            next_token = self.mString[i+1]
        else:
            next_token = 'EOF'
        print('_' * 80)
        print(f'next token = {next_token}')
        print('_' * 80)
        change = True
        while change:
            change = self.complete(i)
            change = self.predict(i) or change
        self.scan(i)
        print(f'\nQ{i}:')
        for obj in self.mStateList[i]:
            print(obj)
        if i + 1 <= n:
            print(f'\nQ{i+1}:')
            for obj in self.mStateList[i+1]:
                print(obj)
    if self.mGrammar.finishObject() in self.mStateList[-1]:
        print('Parsing successful!')
        return True
    print('Parsing failed!')
    return False

EarleyParser.parse = earley_parse
del earley_parse

The method `complete(self, i)` applies the completion operation to the set $Q_i$:
If we have
 - $\langle c \rightarrow \gamma \bullet, j\rangle \in Q_i$  and
 - $\langle a \rightarrow \beta \bullet c \delta, k\rangle \in Q_j$,
then the parser tried to parse the variable $c$ after having read `mString[:j+1]`
and we know that 
$$ c \Rightarrow^* \texttt{mString[j+1:i+1]}, $$
i.e. the parser has recognized $c$ after having read `mString[j+1:i+1]`.
Therefore the parser should proceed to recognize $\delta$ in the set $Q_i$.
Hence we add the *Earley object* $\langle a \rightarrow \beta c \bullet \delta,k\rangle$ to the set $Q_i$:
$$\langle c \rightarrow \gamma \bullet, j\rangle \in Q_i \wedge
  \langle a \rightarrow \beta \bullet c \delta, k\rangle \in Q_j \;\Rightarrow\;
          Q_i := Q_i \cup \bigl\{ \langle a \rightarrow \beta c \bullet \delta, k\rangle \bigr\}
$$

The back pointer
$\bigl(\langle a \rightarrow \beta \bullet c \delta, k\rangle, j, \langle c \rightarrow \gamma \bullet, j\rangle\bigr)$
is added to the list of back pointers of the *Earley object* $\langle a \rightarrow \beta c \bullet \delta,k\rangle$.
In the code, the *Earley object* from $Q_j$ is called `waiting`, since it waits for the variable $c$.  If the
*Earley object* $\langle a \rightarrow \beta c \bullet \delta,k\rangle$ is already an element of $Q_i$, it has been
created before, possibly in a different way.  Then only its list of back pointers is extended, unless it already
contains this back pointer.

The method prints every *Earley object* that it adds to $Q_i$.  Therefore, `newQi` only collects those
*Earley objects* that are not yet elements of $Q_i$, and the loop stops as soon as `newQi` is empty.  The loop
only ends after an iteration that has found no new *Earley object*.  As this iteration inspects all pairs of
*Earley objects*, it finds all back pointers.

In [ ]:
def complete(self, i):
    change = False
    added  = True
    Qi     = self.mStateList[i]
    while added:
        added = False
        newQi: dict[EarleyObject, list[BackPointer]] = {}
        for obj in Qi:
            if obj.isComplete():
                c  = obj.mVariable
                j  = obj.mIndex
                Qj = self.mStateList[j]
                for waiting in Qj:
                    if waiting.sameVar(c):
                        moved = waiting.moveDot()
                        back  = (waiting, j, obj)
                        backs = Qi[moved] if moved in Qi else newQi.setdefault(moved, [])
                        if back not in backs:
                            backs.append(back)
        if newQi:
            change = True
            added  = True
            print('completion:')
            for obj in newQi:
                print(f'{obj} added to Q{i}')
            Qi |= newQi
    return change

EarleyParser.complete = complete
del complete

The method `self.predict(i)` applies the prediction operation to the set $Q_i$: 
If $\langle a \rightarrow \beta \bullet c \delta, k \rangle \in Q_i$, then
the parser tries to recognize $c\delta$ after having read `mString[:i+1]`.  To this end
it has to parse $c$ in the string `mString[i+1:]`.
Therefore, if $c \rightarrow \gamma$ is a rule of our grammar,
we add the *Earley object* $\langle c \rightarrow \bullet \gamma, i\rangle$ to the set $Q_i$:
$$ \langle a \rightarrow \beta \bullet c \delta, k\rangle \in Q_i 
       \wedge (c \rightarrow \gamma) \in R 
       \;\Rightarrow\;
       Q_i := Q_i \cup\bigl\{ \langle c \rightarrow \bullet\gamma, i\rangle\bigr\}.
$$
As the right hand side $\gamma$ might start with a variable, the method uses a fixed-point iteration
until no more *Earley objects* are added to $Q_i$.

The new *Earley objects* get an empty list of back pointers.

The method prints every *Earley object* that it adds to $Q_i$.  Therefore, `newQi` only collects those
*Earley objects* that are not yet elements of $Q_i$, and the loop stops as soon as `newQi` is empty.

In [ ]:
def predict(self, i):
    change = False
    added  = True
    Qi     = self.mStateList[i]
    while added:
        added = False
        newQi: dict[EarleyObject, list[BackPointer]] = {}
        for obj in Qi:
            c = obj.nextVar()
            if c is not None:
                for rule in self.mGrammar.mRules:
                    if c == rule[0]:
                        newObj = EarleyObject(c, (), tuple(rule[1:]), i)
                        if newObj not in Qi:
                            newQi[newObj] = []
        if newQi:
            change = True
            added  = True
            print('prediction:')
            for obj in newQi:
                print(f'{obj} added to Q{i}')
            Qi |= newQi
    return change

EarleyParser.predict = predict
del predict

The method `self.scan(i)` applies the scanning operation to the set $Q_i$.

If $\langle a \rightarrow \beta \bullet T \gamma, k\rangle \in Q_i$ and $T$ is a token,
then the parser tries to recognize the right hand side of the grammar rule
$$ a \rightarrow \beta T \gamma$$ 
and after having read `mString[k+1:i+1]` it has already recognized  $\beta$.
If we now have `mString[i+1] == T`, then the parser still has to recognize $\gamma$ in `mString[i+2:]`.
Therefore, the *Earley object* $\langle a \rightarrow \beta T \bullet \gamma, k\rangle$ is added to
the set $Q_{i+1}$:
$$\langle a \rightarrow \beta \bullet T \gamma, k\rangle \in Q_i \wedge x_{i+1} = T
       \;\Rightarrow\;
       Q_{i+1} := Q_{i+1} \cup \bigl\{ \langle a \rightarrow \beta T \bullet \gamma, k\rangle \bigr\}
$$

The new *Earley object* gets the list containing the back pointer
$\bigl(\langle a \rightarrow \beta \bullet T \gamma, k\rangle, i, T\bigr)$.  This is the only way to create it.

The method prints every *Earley object* that it adds to $Q_{i+1}$.

In [ ]:
def earley_scan(self, i):
    Qi = self.mStateList[i]
    n  = len(self.mString) - 1 # mString[0] = None
    if i + 1 <= n:
        T = self.mString[i+1]
        print('scanning:')
        for obj in Qi:
            if obj.scan(T):
                moved = obj.moveDot()
                print(f'{moved} added to Q{i+1}')
                self.mStateList[i+1][moved] = [(obj, i, T)]

EarleyParser.scan = earley_scan
del earley_scan

## Computing the Parse Trees

If parsing is successful, the set $Q_n$ contains the *Earley object* $\langle\hat{s} \rightarrow s \bullet, 0\rangle$.
Starting with this *Earley object*, the parser follows the back pointers backwards in order to compute the parse
trees.  Whenever an *Earley object* has several back pointers, every one of them leads to different parse trees.

If a variable can derive itself, e.g. with the rule $s \rightarrow s$, a string can have infinitely many parse
trees, since such a rule can be applied any number of times.  Then the back pointers contain a cycle:  The parse tree
of a complete *Earley object* would contain the parse tree of the same *Earley object*.  In order to avoid an
infinite recursion, the methods below get an argument `active` of type `Active`.  It is the set of the pairs
`(obj, i)` such that the parse trees of the complete *Earley object* `obj` in $Q_i$ are being computed at the
moment.  A back pointer that leads to one of these *Earley objects* is skipped.  As a complete *Earley object* in
$Q_i$ stands for a grammar rule that has been used for the part of the input that ends with the token number $i$,
the methods compute all parse trees in which no path from the root to a leaf uses the same grammar rule twice for
the same part of the input.  For the rule $s \rightarrow s$, this means that the rule is used at most once in a row.

**Method `children(self, obj, i, active)`**
- *Input:*
  - an *Earley object* `obj` of the form $\langle a \rightarrow \alpha \bullet \beta, k\rangle$ that is an element of
    the set $Q_i$,
  - the set `active` of the complete *Earley objects* whose parse trees are being computed.
- *Output:* the list of all possible lists of parse trees of the symbols of $\alpha$.

If the list of back pointers of `obj` is empty, then $\alpha$ is empty and there is exactly one list of parse
trees, namely the empty list.  Otherwise, every back pointer is a triple `(pred, j, X)`, where `pred` is an
element of $Q_j$ of the form $\langle a \rightarrow \alpha' \bullet X \beta, k\rangle$ with $\alpha = \alpha' X$.
If `X` is a token, it is its own parse tree.  Otherwise, `X` is a complete *Earley object* in $Q_i$, and its parse
trees are computed by the method `trees`.  The lists of parse trees of the symbols of $\alpha'$ are computed
recursively from `pred`.  Every one of these lists is combined with every parse tree of `X`.

In [ ]:
def children(self, obj: EarleyObject, i: int, active: Active) -> list[list[ParseTree]]:
    backs = self.mStateList[i][obj]
    if backs == []:
        return [[]]
    result: list[list[ParseTree]] = []
    for pred, j, X in backs:
        lasts: list[ParseTree]
        if isinstance(X, str):
            lasts = [X]
        elif (X, i) in active:
            continue
        else:
            lasts = self.trees(X, i, active)
        for prefix in self.children(pred, j, active):
            for last in lasts:
                result.append(prefix + [last])
    return result

EarleyParser.children = children
del children

**Method `trees(self, obj, i, active)`**
- *Input:*
  - a complete *Earley object* `obj` of the form $\langle c \rightarrow \gamma \bullet, j\rangle$ that is an element
    of the set $Q_i$,
  - the set `active` of the complete *Earley objects* whose parse trees are being computed.
- *Output:* the list of all parse trees of the variable $c$ that `obj` has recognized.

The pair `(obj, i)` is added to `active` while the parse trees of `obj` are computed.  If $\gamma$ is empty, the
variable $c$ gets the single child `'λ'`.

In [ ]:
def trees(self, obj: EarleyObject, i: int, active: Active) -> list[ParseTree]:
    active = active | {(obj, i)}
    result: list[ParseTree] = []
    for subtrees in self.children(obj, i, active):
        if subtrees == []:
            result.append((obj.mVariable, 'λ'))
        else:
            result.append((obj.mVariable, *subtrees))
    return result

EarleyParser.trees = trees
del trees

**Method `parseTrees(self)`**
- *Input:* none.  It is assumed that parsing has been successful.
- *Output:* the list of all parse trees of the string that has been parsed.

The *Earley object* $\langle\hat{s} \rightarrow s \bullet, 0\rangle \in Q_n$ has a single child, namely a parse
tree of the start variable $s$.  Every such child is a parse tree of the string.

In [ ]:
def parseTrees(self) -> list[ParseTree]:
    n = len(self.mString) - 1 # mString[0] = None
    return [subtrees[0] for subtrees in self.children(self.mGrammar.finishObject(), n, frozenset())]

EarleyParser.parseTrees = parseTrees
del parseTrees

The function `tokenize` transforms the string `s` that is to be parsed into a list of tokens.  Parentheses and
operators are returned as literals, i.e. enclosed in double quotes, since the grammar writes them this way.
See below for an example.

In [ ]:
def tokenize(s):
    '''Transform the string s into a list of tokens.  The string s
       is supposed to represent an arithmetic expression.
    '''
    lexSpec = r'''([ \t]+)        |  # blanks and tabs
                  ([1-9][0-9]*|0) |  # number
                  ([()])          |  # parentheses 
                  ([-+*/])        |  # arithmetical operators
                  (.)                # unrecognized character
               '''
    tokenList = re.findall(lexSpec, s, re.VERBOSE)
    result    = []
    for ws, number, parenthesis, operator, error in tokenList:
        if ws:        # skip blanks and tabs
            continue
        elif number:
            result += [ 'NUMBER' ]
        elif parenthesis:
            result += [ f'"{parenthesis}"' ]
        elif operator:
            result += [ f'"{operator}"' ]
        else:
            result += [ f'ERROR({error})']
    return result

In [ ]:
tokenize('1 + 2 * 3')

The function `test` takes two arguments.
- `file` is the name of a file containing a grammar,
- `word` is a string that should be parsed.

First, the function `read_grammar` reads the grammar from `file` and converts it into a list of rules, which is
passed to the constructor of the class `Grammar`.  Then `word` is tokenized.  Finally, `word` is parsed using
*Earley's algorithm*.  If parsing is successful, the function prints the number of parse trees and draws
every parse tree with the function `tuple2dot`.

In [ ]:
def test(file, word):
    grammar   = Grammar(read_grammar(file))
    TokenList = tokenize(word)
    ep        = EarleyParser(grammar, TokenList)
    if ep.parse():
        trees = ep.parseTrees()
        print(f'number of parse trees: {len(trees)}')
        for tree in trees:
            display(tuple2dot(tree))

In [ ]:
test('simple.g', '1 + 2 * 3')

The grammar in the file `lambda.g` describes the same expressions as the grammar in `simple.g`, but it avoids
left recursion.  Therefore, it needs $\lambda$-rules, i.e. rules with an empty right hand side.  For such a
grammar, the method `parse` has to repeat the completion and the prediction operation until the set $Q_i$ does
not change any more.  The output shows how completion and prediction alternate in the same set $Q_i$.

In [ ]:
Code(filename='lambda.g', language='larkita')

In [ ]:
test('lambda.g', '1 * 2')

The grammar in the file `ambiguous.g` is ambiguous, since it does not specify the precedence of the operators.
Therefore, the string `1 + 2 * 3` has two parse trees:  In one of them, the root uses the operator `+`, i.e. the
string is read as `1 + (2 * 3)`.  In the other one, the root uses the operator `*`, i.e. the string is read as
`(1 + 2) * 3`.

In [ ]:
Code(filename='ambiguous.g', language='larkita')

In [ ]:
test('ambiguous.g', '1 + 2 * 3')